# Audio Generasiyası — Hugging Face (Transformers)

Bu notebook iki fərqli audio generasiyasını göstərir: **nitq (text-to-speech)** və **musiqi (text-to-music)**.

## Bu iki tapşırıq nə ilə fərqlənir?

- **Text-to-speech (TTS):** yazılı mətni səsli nitqə çevirir — köməkçi tətbiqlər, audiokitab, dublyaj üçün
- **Text-to-music:** mətn təsvirinə (janr, alət, ovqat) əsasən musiqi parçası yaradır — hazır melodiya yoxdur, model onu "təxəyyül edir"

Hər ikisi eyni `pipeline` interfeysi ilə işləyir, sadəcə model və çıxış fərqlidir.

> **Colab GPU:** İşə başlamazdan əvvəl: `Runtime → Change runtime type → GPU (T4)` seç. GPU olmadan bu modellər ya işləməyəcək, ya da çox yavaş olacaq.

Bu modellər (small versiyalar) kiçikdir, CPU-da da işləyər, amma GPU ilə xeyli sürətlidir.

In [ ]:
!pip install -q transformers accelerate scipy

In [ ]:
from transformers import pipeline
from IPython.display import Audio
import scipy

## 1. Nitq (Text-to-Speech)
`suno/bark-small` — kiçik, sürətli, çoxdilli dəstəyi olan TTS modeli.

In [ ]:
tts = pipeline("text-to-speech", model="suno/bark-small")

speech = tts("Salam, bu Hugging Face ilə generasiya olunan səsdir.")

scipy.io.wavfile.write("speech.wav", rate=speech["sampling_rate"], data=speech["audio"])
Audio("speech.wav")

### Fərqli mətnlər sınayaq

In [ ]:
for text in [
    "Welcome to this text to speech demo.",
    "Bu model səsi necə yaradır, maraqlıdır, elə deyilmi?",
]:
    result = tts(text)
    display(Audio(result["audio"], rate=result["sampling_rate"]))

## 2. Musiqi (Text-to-Music)
`facebook/musicgen-small` (300M) — free T4-də rahat işləyən musiqi generasiya modeli.

In [ ]:
musicgen = pipeline("text-to-audio", "facebook/musicgen-small")

music = musicgen(
    "lo-fi hip hop beat with soft piano, relaxing",
    forward_params={"do_sample": True}
)

scipy.io.wavfile.write("music.wav", rate=music["sampling_rate"], data=music["audio"])
Audio("music.wav")

### Parametrlər nə deməkdir?

- `do_sample=True` — hər dəfə bir az fərqli nəticə (sampling), `False` olsa daha "təhlükəsiz"/təkrarlanan nəticə
- `max_new_tokens` (processor + model yolu ilə) — musiqinin uzunluğunu idarə edir, `pipeline` default-u ~5 saniyədir

## Növbəti addımlar

- Daha keyfiyyətli səs üçün `facebook/musicgen-medium` və ya `-large` sına (daha çox VRAM istəyir)
- `microsoft/speecht5_tts` kimi modellərlə fərqli səs "timbrləri" (speaker embedding) sınamaq olar
- Bark modelinin böyük versiyası (`suno/bark`) daha ifadəli intonasiya verir, amma yavaşdır